# Machine Learning — Lab 7
## Classification Evaluation, Thresholds, and Model Selection

**Main Course Learning Outcomes — CLO4 and CLO5**

- **CLO4:** Apply supervised learning techniques to solve practical problems and interpret their outcomes.
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** A classifier should not be judged by accuracy alone. Full credit requires you to explain **which mistakes matter, how threshold changes affect decisions, how class imbalance changes interpretation, and why model selection must use validation evidence rather than the final test set**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Confusion-matrix fundamentals | 20 min | Compute TP, TN, FP, FN and core metrics |
| 2. Class imbalance | 15 min | Show why accuracy can be misleading |
| 3. Threshold experiments | 25 min | Observe precision/recall/F1 trade-offs |
| 4. Cost-sensitive evaluation | 20 min | Compare models using application costs |
| 5. Cross-validation & model selection | 25 min | Select hyperparameters without touching the test set |
| 6. Challenge, debugging & viva | 15 min | Diagnose evaluation mistakes and defend a model choice |
| **Total** | **120 min** | |

### Main idea

$$
\boxed{
\text{Predicted probabilities}
\rightarrow
\text{Threshold}
\rightarrow
\text{Confusion matrix}
\rightarrow
\text{Metrics}
\rightarrow
\text{Decision}
}
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. construct and interpret a confusion matrix;
2. compute accuracy, precision, recall, specificity, and $F_1$;
3. explain why accuracy may be misleading on imbalanced data;
4. predict how threshold changes affect FP and FN;
5. compare classifiers using precision-recall trade-offs;
6. apply a cost-sensitive decision rule;
7. use cross-validation for hyperparameter selection;
8. distinguish validation selection from final test evaluation;
9. compare two classifiers using both metrics and context;
10. justify a final model choice clearly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

print("Machine Learning Lab 7 environment ready.")

# Part I — Dataset and Evaluation Context

We will simulate a **fraud-detection dataset**.

The positive class is:

$$
y=1 \quad \text{Fraud}
$$

The negative class is:

$$
y=0 \quad \text{Legitimate}
$$

This is intentionally imbalanced because fraud is relatively rare.

The goal is not simply to maximize overall accuracy. Missing fraud may be much more costly than temporarily flagging a legitimate transaction.

In [ ]:
X_arr, y_arr = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_repeated=0,
    n_clusters_per_class=2,
    weights=[0.90, 0.10],
    class_sep=1.1,
    flip_y=0.02,
    random_state=3452,
)

feature_names = [f"feature_{i+1}" for i in range(X_arr.shape[1])]

df_master = pd.DataFrame(X_arr, columns=feature_names)
df_master["fraud"] = y_arr

print("Master dataset shape:", df_master.shape)
print("\nClass counts:")
display(df_master["fraud"].value_counts().rename(index={0:"legitimate", 1:"fraud"}))

## Task 1.1 — Personalized Working Dataset

Enter the last four digits of your student ID.

Your value determines a reproducible sample of 900 transactions.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 7000 + (STUDENT_ID_LAST4 % 3000)

df = df_master.sample(
    n=900,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your evaluation seed:", SEED)
print("Working dataset shape:", df.shape)

In [ ]:
X = df[feature_names].copy()
y = df["fraud"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=SEED,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

print("\nValidation class proportions:")
display(y_valid.value_counts(normalize=True).sort_index().rename("proportion"))

## Task 1.2 — Predict Before Modeling

Before fitting a classifier, answer:

1. Which class is the minority class?
2. If we always predict `legitimate`, what accuracy do you expect approximately?
3. Would that baseline detect fraud?
4. Why is this a warning against using accuracy alone?

# Part II — Confusion Matrix Fundamentals

For binary classification:

| | Predicted Positive | Predicted Negative |
|---|---:|---:|
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

For this lab:

- TP = fraud correctly detected
- FN = fraud missed
- FP = legitimate transaction incorrectly flagged
- TN = legitimate transaction correctly accepted

## Task 2.1 — Manual Metric Formulas

Write the formulas before using code:

$$
Accuracy = \ ?
$$

$$
Precision = \ ?
$$

$$
Recall = \ ?
$$

$$
Specificity = \ ?
$$

$$
F_1 = \ ?
$$

Then explain which metric is most directly affected by **false negatives**.

## Task 2.2 — Implement Metric Computation

Complete the function.

In [ ]:
def metrics_from_counts(tp, tn, fp, fn):
    # TODO: compute all five metrics.
    accuracy = None
    precision = None
    recall = None
    specificity = None
    f1 = None

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
    }

In [ ]:
# Self-check.
check = metrics_from_counts(
    tp=40,
    tn=130,
    fp=10,
    fn=20
)

assert abs(check["accuracy"] - 0.85) < 1e-12
assert abs(check["precision"] - 0.80) < 1e-12
assert abs(check["recall"] - (40/60)) < 1e-12
assert abs(check["specificity"] - (130/140)) < 1e-12

print("Metric function tests passed.")
print(check)

# Part III — Fit a Reference Classifier

We will start with logistic regression.

The model outputs a probability:

$$
p=P(y=1\mid x).
$$

The default decision threshold is often:

$$
t=0.5.
$$

In [ ]:
logistic = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=SEED
    )),
])

logistic.fit(X_train, y_train)

valid_prob = logistic.predict_proba(X_valid)[:, 1]
valid_pred_05 = (valid_prob >= 0.5).astype(int)

cm = confusion_matrix(y_valid, valid_pred_05)
tn, fp, fn, tp = cm.ravel()

print("Validation confusion matrix at threshold 0.5:")
display(pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

print("TP, TN, FP, FN:", tp, tn, fp, fn)

## Task 3.1 — Interpret the Confusion Matrix

Answer:

1. How many fraud cases were correctly detected?
2. How many fraud cases were missed?
3. How many legitimate transactions were incorrectly flagged?
4. Which error type is probably more expensive in this scenario?
5. Why should that influence threshold choice?

In [ ]:
valid_metrics_05 = metrics_from_counts(tp, tn, fp, fn)

print("Metrics at threshold 0.5:")
display(pd.Series(valid_metrics_05).round(4).to_frame("value"))

## Task 3.2 — Compare Accuracy and Recall

Answer:

1. Is accuracy high?
2. Is fraud recall equally high?
3. Why can these two values differ so much?
4. What would happen if the model predicted every case as legitimate?

# Part IV — Threshold Experiments

A threshold converts probabilities into class labels:

$$
\hat{y}=
\begin{cases}
1, & p\ge t\\
0, & p<t.
\end{cases}
$$

Lower threshold:

- more predicted positives;
- usually higher recall;
- usually more false positives.

Higher threshold:

- fewer predicted positives;
- usually higher precision;
- usually more false negatives.

## Task 4.1 — Predict Before Running

For thresholds:

$$
0.2,\;0.3,\;0.5,\;0.7
$$

predict which threshold will likely have:

- highest recall;
- highest precision;
- most false positives;
- most false negatives.

**Your prediction:**

In [ ]:
thresholds = [0.2, 0.3, 0.5, 0.7]
rows = []

for t in thresholds:
    pred = (valid_prob >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_valid, pred).ravel()

    m = metrics_from_counts(tp_t, tn_t, fp_t, fn_t)

    rows.append({
        "threshold": t,
        "TP": tp_t,
        "FP": fp_t,
        "FN": fn_t,
        "TN": tn_t,
        **m,
    })

threshold_table = pd.DataFrame(rows)
display(threshold_table.round(4))

## Task 4.2 — Analyze Threshold Trade-offs

Answer:

1. Which threshold achieved the highest recall?
2. Which threshold achieved the highest precision?
3. Which threshold produced the fewest false negatives?
4. Which threshold produced the most false positives?
5. Did the threshold with highest accuracy also have highest recall?
6. Why is there no universally best threshold?

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    threshold_table["threshold"],
    threshold_table["precision"],
    marker="o",
    label="Precision"
)
plt.plot(
    threshold_table["threshold"],
    threshold_table["recall"],
    marker="o",
    label="Recall"
)
plt.plot(
    threshold_table["threshold"],
    threshold_table["f1"],
    marker="o",
    label="F1"
)
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.title("Threshold Trade-offs")
plt.legend()
plt.show()

## Task 4.3 — Explain the Plot

Describe:

- what happens to recall as threshold increases;
- what happens to precision;
- whether $F_1$ peaks at an intermediate threshold;
- why the best threshold depends on the application rather than the plot alone.

# Part V — Precision–Recall Curve

Rather than testing only four thresholds, we can evaluate many thresholds.

A precision–recall curve is particularly useful when the positive class is rare.

In [ ]:
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(
    y_valid,
    valid_prob
)

plt.figure(figsize=(7, 5))
plt.plot(recall_vals, precision_vals)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.show()

## Task 5.1 — Interpret the Curve

Answer:

1. What happens to precision when we try to capture almost all positives?
2. Why is this curve useful for imbalanced classification?
3. Why does it not directly tell us the business cost of a threshold?
4. What extra information is needed to make an operational decision?

# Part VI — Cost-Sensitive Evaluation

Suppose:

- each false negative costs **500 SAR**;
- each false positive costs **10 SAR**.

Then:

$$
Cost = 500(FN)+10(FP).
$$

This cost structure strongly penalizes missed fraud.

## Task 6.1 — Predict Before Computing

Which threshold do you expect to have the lowest cost:

- 0.2
- 0.3
- 0.5
- 0.7

Explain your reasoning before running the next cell.

In [ ]:
FN_COST = 500
FP_COST = 10

cost_rows = []

for _, row in threshold_table.iterrows():
    cost = FN_COST * row["FN"] + FP_COST * row["FP"]

    cost_rows.append({
        "threshold": row["threshold"],
        "FP": int(row["FP"]),
        "FN": int(row["FN"]),
        "cost": cost,
    })

cost_table = pd.DataFrame(cost_rows)
display(cost_table.sort_values("cost"))

## Task 6.2 — Interpret Cost-Sensitive Results

Answer:

1. Which threshold has the lowest cost?
2. Does it also have the highest accuracy?
3. Why can a lower-accuracy threshold be better?
4. What assumption is built into the values 500 SAR and 10 SAR?
5. How should those costs be chosen in a real organization?

# Part VII — Compare Two Classifiers

We will compare:

- Logistic Regression
- K-Nearest Neighbors

The goal is not to declare one algorithm universally better.

We compare them using validation evidence.

In [ ]:
knn = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7)),
])

knn.fit(X_train, y_train)

knn_valid_prob = knn.predict_proba(X_valid)[:, 1]
knn_valid_pred = (knn_valid_prob >= 0.5).astype(int)

def evaluate_classifier(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    m = metrics_from_counts(tp, tn, fp, fn)
    m["roc_auc"] = roc_auc_score(y_true, prob)
    m["FP"] = fp
    m["FN"] = fn
    return m

comparison = pd.DataFrame({
    "Logistic": evaluate_classifier(y_valid, valid_prob, 0.5),
    "KNN_k7": evaluate_classifier(y_valid, knn_valid_prob, 0.5),
}).T

display(comparison.round(4))

## Task 7.1 — Compare the Models

Answer:

1. Which model has higher accuracy?
2. Which has higher recall?
3. Which has higher precision?
4. Which has higher ROC AUC?
5. Which model would you prefer if missing fraud is very expensive?
6. Which model would you prefer if manual review capacity is very limited?

# Part VIII — Cross-Validation for Hyperparameter Selection

We now choose the number of neighbors $k$ for KNN.

Candidate values:

$$
k\in\{1,3,5,7,11,15\}.
$$

We will use **Stratified 5-fold cross-validation** on the training set.

The validation and test sets remain untouched during this search.

## Task 8.1 — Why Cross-Validation?

Explain:

1. Why one training/validation split can be noisy.
2. What 5-fold cross-validation does.
3. Why stratification matters here.
4. Why the test set must remain untouched.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

k_values = [1, 3, 5, 7, 11, 15]
cv_rows = []

for k in k_values:
    model = Pipeline(steps=[
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"
    )

    cv_rows.append({
        "k": k,
        "mean_cv_f1": scores.mean(),
        "std_cv_f1": scores.std(),
    })

cv_table = pd.DataFrame(cv_rows)
display(cv_table.round(4))

## Task 8.2 — Select $k$

Answer:

1. Which $k$ has the highest mean cross-validation $F_1$?
2. Which $k$ has the most variable fold scores?
3. Why might $k=1$ overfit?
4. Why might a very large $k$ underfit?
5. Why did we use $F_1$ rather than accuracy for this imbalanced task?

In [ ]:
best_k = int(
    cv_table.loc[
        cv_table["mean_cv_f1"].idxmax(),
        "k"
    ]
)

print("Selected k from training-set cross-validation:", best_k)

knn_selected = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
])

knn_selected.fit(X_train, y_train)

selected_valid_prob = knn_selected.predict_proba(X_valid)[:, 1]
selected_valid_eval = evaluate_classifier(
    y_valid,
    selected_valid_prob,
    threshold=0.5
)

print("\nSelected KNN validation performance:")
display(pd.Series(selected_valid_eval).round(4).to_frame("value"))

# Part IX — Validation-Based Final Selection

We now compare:

- Logistic Regression
- Cross-validation-selected KNN

Use validation performance and application context.

Do **not** inspect the test set yet.

In [ ]:
final_validation_comparison = pd.DataFrame({
    "Logistic": evaluate_classifier(y_valid, valid_prob, 0.5),
    f"KNN_k{best_k}": selected_valid_eval,
}).T

display(final_validation_comparison.round(4))

## Task 9.1 — Choose the Final Model

Make a decision under this application rule:

> Missing fraud is more serious than flagging some legitimate transactions, but manual review capacity is not unlimited.

State:

1. which model you choose;
2. which metrics most influenced the choice;
3. whether you would keep threshold 0.5;
4. what additional threshold experiment you would run;
5. why you are not selecting by accuracy alone.

# Part X — Personalized Threshold Challenge

Your student-ID seed assigns one threshold.

You must predict its behavior before calculating the metrics.

In [ ]:
personal_thresholds = [0.15, 0.25, 0.35, 0.45, 0.60, 0.75]
PERSONAL_THRESHOLD = personal_thresholds[
    SEED % len(personal_thresholds)
]

print("Your assigned threshold:", PERSONAL_THRESHOLD)

## Task 10.1 — Predict Before Running

Relative to threshold 0.5, predict:

- recall: increase / decrease / similar
- precision: increase / decrease / similar
- FP: increase / decrease / similar
- FN: increase / decrease / similar

Explain why.

In [ ]:
personal_eval = evaluate_classifier(
    y_valid,
    valid_prob,
    threshold=PERSONAL_THRESHOLD
)

print("Threshold 0.5:")
display(pd.Series(evaluate_classifier(y_valid, valid_prob, 0.5)).round(4).to_frame("value"))

print("\nYour threshold:")
display(pd.Series(personal_eval).round(4).to_frame("value"))

## Task 10.2 — Analyze Your Threshold

Answer:

1. Was your prediction correct?
2. Which metric changed most?
3. How many more/fewer false positives occurred?
4. How many more/fewer false negatives occurred?
5. Under the cost rule $500(FN)+10(FP)$, is your threshold better or worse than 0.5?

# Part XI — Deliberate Debugging

A student writes:

```python
best_model = max(models, key=lambda m: test_accuracy[m])
```

They use the final test set to choose the model.

Another student reports only:

```text
Accuracy = 92%
```

for an imbalanced fraud dataset.

Both evaluation practices are flawed.

## Task 11.1 — Diagnose the Two Errors

Explain:

1. Why selecting by test accuracy contaminates the final evaluation.
2. Which data should select the model instead.
3. Why 92% accuracy may be poor in a dataset with 90% legitimate transactions.
4. Which additional metrics should be reported.

## Task 11.2 — Fix a Broken Recall Function

The function below is incorrect:

```python
def bad_recall(tp, fp, fn):
    return tp / (tp + fp)
```

Explain the mistake, then complete:

In [ ]:
def correct_recall(tp, fn):
    # TODO: implement recall.
    return None

In [ ]:
assert abs(correct_recall(40, 20) - (2/3)) < 1e-12
print("Recall debugging test passed.")

# Part XII — Final Test Evaluation

After model and threshold decisions are finalized, evaluate once on the test set.

For consistency, the code below uses the validation-selected KNN model at threshold 0.5 and the logistic model at threshold 0.5.

You may report both, but do not tune anything after seeing these results.

In [ ]:
log_test_prob = logistic.predict_proba(X_test)[:, 1]
knn_test_prob = knn_selected.predict_proba(X_test)[:, 1]

test_comparison = pd.DataFrame({
    "Logistic": evaluate_classifier(y_test, log_test_prob, 0.5),
    f"KNN_k{best_k}": evaluate_classifier(y_test, knn_test_prob, 0.5),
}).T

display(test_comparison.round(4))

## Task 12.1 — Final Generalization Statement

Write 4–6 sentences that include:

- which model you selected;
- the validation evidence used;
- final test accuracy;
- final test precision;
- final test recall;
- final test $F_1$;
- whether test behavior is reasonably consistent with validation;
- one limitation of using a fixed threshold 0.5.

Do not change the model or threshold after seeing the test metrics.

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. Explain TP, TN, FP, and FN in the fraud context.
2. Why can accuracy be misleading when classes are imbalanced?
3. What is the difference between precision and recall?
4. Why does lowering the threshold usually increase recall?
5. Why can a lower-accuracy model be better under asymmetric error costs?
6. Why is cross-validation used for model selection?
7. Why must the final test set remain untouched?
8. For your personalized threshold, explain exactly why the metrics changed.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. Which metric was most important in this fraud scenario and why?
2. What did threshold experiments teach you that accuracy alone could not?
3. Why is $F_1$ useful but still not enough for every application?
4. What is the purpose of cost-sensitive evaluation?
5. What is the most important rule for model selection and final testing?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] your own student-ID-derived sample;
- [ ] confusion-matrix interpretation;
- [ ] completed manual metric function;
- [ ] logistic-regression validation metrics;
- [ ] threshold experiment;
- [ ] precision-recall interpretation;
- [ ] cost-sensitive evaluation;
- [ ] Logistic vs. KNN comparison;
- [ ] 5-fold cross-validation for KNN;
- [ ] validation-based final model choice;
- [ ] personalized threshold analysis;
- [ ] corrected recall debugging task;
- [ ] final test evaluation;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\boxed{
\text{probability}
\rightarrow
\text{threshold}
\rightarrow
\text{error types}
\rightarrow
\text{metrics}
\rightarrow
\text{contextual decision}
}
$$

# Lab 7 Summary

You should now be able to evaluate classification systems beyond accuracy.

### Confusion-matrix metrics

$$
Accuracy=\frac{TP+TN}{TP+TN+FP+FN}
$$

$$
Precision=\frac{TP}{TP+FP}
$$

$$
Recall=\frac{TP}{TP+FN}
$$

$$
Specificity=\frac{TN}{TN+FP}
$$

$$
F_1=
2\frac{Precision\times Recall}{Precision+Recall}
$$

### Main lessons

- class imbalance can make accuracy misleading;
- thresholds change the FP/FN trade-off;
- the best threshold depends on application costs;
- cross-validation helps select hyperparameters;
- validation selects models;
- the test set evaluates the final decision once.

**Next lab:** Decision Trees — Splitting, Entropy, and Pruning.